### 1. Imports and Configuration

In [ ]:
import sys
import os
import time
import signal
import logging
import subprocess
from pathlib import Path

from google.adk.agents                    import Agent
from google.adk.agents.remote_a2a_agent   import RemoteA2aAgent
from google.adk.sessions                  import InMemorySessionService
from google.adk.runners                   import Runner
from google.genai                         import types
from IPython.display                      import display, Markdown

# -----------------------------------------------------------------------
# Path setup 
# -----------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import APP_NAME, USER_ID, strip_emojis

logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt = "%H:%M:%S",
)
logging.getLogger("google.genai").setLevel(logging.ERROR)
log = logging.getLogger(__name__)

### 2. A2A Server Management

The three financial specialist agents are defined in `agents/`. Each is a
standalone Python module that wraps an ADK agent with `to_a2a()` and exposes
a uvicorn-compatible ASGI app. We start them as subprocesses from the notebook
and shut them down when finished.

In [ ]:
# -----------------------------------------------------------------------
# Start the three A2A agent servers as background subprocesses.
# Each runs on its own port.
# -----------------------------------------------------------------------

AGENTS = {
    "mortgage":   {"module": "agents.mortgage_agent:a2a_app",   "port": 8001},
    "risk":       {"module": "agents.risk_agent:a2a_app",       "port": 8002},
    "investment": {"module": "agents.investment_agent:a2a_app",  "port": 8003},
}

processes = {}

for name, cfg in AGENTS.items():
    cmd = [
        sys.executable, "-m", "uvicorn",
        cfg["module"],
        "--host", "localhost",
        "--port", str(cfg["port"]),
        "--log-level", "warning",
    ]
    proc = subprocess.Popen(
        cmd,
        cwd    = str(PROJECT_ROOT),
        stdout = subprocess.PIPE,
        stderr = subprocess.PIPE,
    )
    processes[name] = proc
    log.info("started %s agent on port %d (pid=%d)", name, cfg["port"], proc.pid)

# Give servers time to start
time.sleep(5)
log.info("A2A servers ready.")

05:37:53  INFO      started mortgage agent on port 8001 (pid=9439)
05:37:53  INFO      started risk agent on port 8002 (pid=9440)
05:37:53  INFO      started investment agent on port 8003 (pid=9441)
05:37:58  INFO      All A2A ready.


In [4]:
for name, proc in processes.items():
    if proc.poll() is not None:
        stderr = proc.stderr.read().decode()
        log.error("%s exited with code %d: %s", name, proc.returncode, stderr[-500:])
    else:
        log.info("%s still running (pid=%d)", name, proc.pid)

05:37:58  INFO      mortgage still running (pid=9439)
05:37:58  INFO      risk still running (pid=9440)
05:37:58  INFO      investment still running (pid=9441)


### 3. Verify Agent Cards

Each A2A server auto-generates an agent card at its URL. Fetching these confirms the servers are running and the agents are properly described.

In [5]:
import httpx

for name, cfg in AGENTS.items():
    url = f"http://localhost:{cfg['port']}/.well-known/agent-card.json"
    try:
        r = httpx.get(url, timeout=5)
        card = r.json()
        log.info("%s agent card: name=%s, description=%s",
                 name, card.get("name"), card.get("description", "")[:80])
    except Exception as e:
        log.error("%s agent card fetch failed: %s", name, e)

05:37:58  INFO      HTTP Request: GET http://localhost:8001/.well-known/agent-card.json "HTTP/1.1 200 OK"
05:37:58  INFO      mortgage agent card: name=mortgage_agent, description=Calculates mortgage payments, amortization schedules, and affordability.
05:37:58  INFO      HTTP Request: GET http://localhost:8002/.well-known/agent-card.json "HTTP/1.1 200 OK"
05:37:58  INFO      risk agent card: name=risk_agent, description=Evaluates credit risk scores and performs loan risk assessments.
05:37:58  INFO      HTTP Request: GET http://localhost:8003/.well-known/agent-card.json "HTTP/1.1 200 OK"
05:37:58  INFO      investment agent card: name=investment_agent, description=Projects investment growth, calculates ROI, and plans savings goals.


### 4. Run Helper

In [6]:
# -----------------------------------------------------------------------
# Same pattern as notebooks 2 and 3 but using make_runner
# directly since config.py MODEL is not used here (the coordinator
# uses RemoteA2aAgent, not a local model for the specialists).
# -----------------------------------------------------------------------

def make_runner(agent):
    return Runner(
        agent           = agent,
        app_name        = APP_NAME,
        session_service = InMemorySessionService(),
    )

async def run_agent(agent, query):
    """Send a query and return the final response."""
    runner  = make_runner(agent)
    session = await runner.session_service.create_session(
        app_name = APP_NAME,
        user_id  = USER_ID,
    )
    content = types.Content(
        role  = "user",
        parts = [types.Part(text=query)],
    )

    final_text   = None
    final_author = None

    async for event in runner.run_async(
        user_id     = USER_ID,
        session_id  = session.id,
        new_message = content,
    ):
        author = getattr(event, "author", "unknown")

        if event.content and event.content.parts:
            for part in event.content.parts:
                if hasattr(part, "function_call") and part.function_call:
                    log.info("[%s] tool_call: %s(%s)",
                             author, part.function_call.name,
                             dict(part.function_call.args or {}))
                if hasattr(part, "function_response") and part.function_response:
                    log.info("[%s] tool_response: %s -> %s",
                             author, part.function_response.name,
                             str(part.function_response.response)[:200])

        if event.is_final_response() and event.content and event.content.parts:
            for part in event.content.parts:
                if not hasattr(part, "text") or not part.text:
                    continue
                if not getattr(part, "thought", False):
                    final_text   = strip_emojis(part.text)
                    final_author = author

    if final_text:
        display(Markdown(f"**{final_author}:**\n\n{final_text}"))

    return final_text

### 5. Remote Agents + Coordinator

The coordinator is a local agent that uses `RemoteA2aAgent` to connect
to the three A2A servers. From the coordinator's perspective, they look
like local sub-agents, but communication goes over HTTP via the A2A protocol.

In [ ]:
from google.adk.agents.remote_a2a_agent import AGENT_CARD_WELL_KNOWN_PATH

# -----------------------------------------------------------------------
# Connect to the three remote A2A agents via their agent card URLs.
# -----------------------------------------------------------------------

remote_mortgage = RemoteA2aAgent(
    name        = "mortgage_agent",
    description = "Remote A2A agent for mortgage calculations.",
    agent_card  = f"http://localhost:8001/.well-known/agent-card.json",
)

remote_risk = RemoteA2aAgent(
    name        = "risk_agent",
    description = "Remote A2A agent for credit risk and loan risk assessment.",
    agent_card  = f"http://localhost:8002/.well-known/agent-card.json",
)

remote_investment = RemoteA2aAgent(
    name        = "investment_agent",
    description = "Remote A2A agent for investment projections and ROI.",
    agent_card  = f"http://localhost:8003/.well-known/agent-card.json",
)

# -----------------------------------------------------------------------
# Coordinator agent -- routes queries to the correct remote specialist
# via transfer_to_agent (same pattern as notebook 3 section 3, but
# the specialists are now separate network services).
# -----------------------------------------------------------------------

a2a_coordinator = Agent(
    model       = "gemini-3-flash-preview",
    name        = "a2a_coordinator",
    description = "Routes financial queries to remote A2A specialist agents.",
    instruction = (
        "You are a financial services coordinator. Based on the user's question, "
        "delegate to the appropriate remote specialist:\n"
        "- mortgage_agent for payment calculations, amortization, affordability\n"
        "- risk_agent for credit risk or loan risk evaluation\n"
        "- investment_agent for compound interest, ROI, savings goals\n"
        "Transfer to the best-fit specialist. Do not answer directly."
    ),
    sub_agents = [remote_mortgage, remote_risk, remote_investment],
)

/var/folders/vx/22chbmvd0fbdfysfq_h_8_sr0000gn/T/ipykernel_9339/1131359649.py:7: UserWarning: [EXPERIMENTAL] RemoteA2aAgent: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subject to breaking changes. A2A protocol and SDK are themselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  remote_mortgage = RemoteA2aAgent(
/var/folders/vx/22chbmvd0fbdfysfq_h_8_sr0000gn/T/ipykernel_9339/1131359649.py:13: UserWarning: [EXPERIMENTAL] RemoteA2aAgent: ADK Implementation for A2A support (A2aAgentExecutor, RemoteA2aAgent and corresponding supporting components etc.) is in experimental mode and is subject to breaking changes. A2A protocol and SDK are themselves not experimental. Once it's stable enough the experimental mode will be removed. Your feedback is welcome.
  remote_risk = RemoteA2aAgent(
/var/folders/vx/22chbmvd0fbdfysfq_h_8_sr0000g

### 6. Test Queries

These are the same queries from notebook 3 section 3, but now the
specialists are running as separate A2A services rather than in-process.

In [8]:
# -----------------------------------------------------------------------
# Mortgage query -- should route to the remote mortgage_agent.
# -----------------------------------------------------------------------

response = await run_agent(
    a2a_coordinator,
    "How much can I afford if I make 140k and have 800/mo in debts? Rate is 5.75%.",
)

05:38:06  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
05:38:08  INFO      Response received from the model.
05:38:08  WARNING   Warning: there are non-text parts in the response: ['function_call'], returning concatenated text result from text parts. Check the full candidates.content.parts accessor to get the full model response.
05:38:08  INFO      [a2a_coordinator] tool_call: transfer_to_agent({'agent_name': 'mortgage_agent'})
05:38:08  INFO      [a2a_coordinator] tool_response: transfer_to_agent -> {'result': None}
05:38:08  INFO      HTTP Request: GET http://localhost:8001/.well-known/agent-card.json "HTTP/1.1 200 OK"
05:38:08  INFO      Successfully fetched agent card data from http://localhost:8001/.well-known/agent-card.json: {'capabilities': {}, 'defaultInputModes': ['text/plain'], 'defaultOutputModes': ['text/plain'], 'description': 'Calculates mortgage payments, amortization schedules, and affordability.', '

**mortgage_agent:**

Based on an annual income of **$140,000**, monthly debts of **$800**, and a **5.75%** interest rate, here is an estimate of what you can afford:

*   **Maximum Home Price:** $722,560
*   **Maximum Monthly Mortgage Payment (P&I):** $4,216.67
*   **Total Monthly Debt (including mortgage):** $5,016.67
*   **Debt-to-Income (DTI) Ratio:** 43%

This estimate assumes a **30-year term** and a **$0 down payment**. If you have a down payment available, your maximum home price would increase by that amount. Keep in mind that this calculation covers principal and interest; you should also factor in property taxes, homeowners insurance, and potential HOA fees.

In [9]:
# -----------------------------------------------------------------------
# Risk query -- should route to the remote risk_agent.
# -----------------------------------------------------------------------

response = await run_agent(
    a2a_coordinator,
    "Score my credit risk: 680 credit score, 42% DTI, 0.92 LTV, "
    "2 years employed, 1 missed payment.",
)

05:40:01  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
05:40:03  INFO      Response received from the model.
05:40:03  INFO      [a2a_coordinator] tool_call: transfer_to_agent({'agent_name': 'risk_agent'})
05:40:03  INFO      [a2a_coordinator] tool_response: transfer_to_agent -> {'result': None}
05:40:03  INFO      HTTP Request: GET http://localhost:8002/.well-known/agent-card.json "HTTP/1.1 200 OK"
05:40:03  INFO      Successfully fetched agent card data from http://localhost:8002/.well-known/agent-card.json: {'capabilities': {}, 'defaultInputModes': ['text/plain'], 'defaultOutputModes': ['text/plain'], 'description': 'Evaluates credit risk scores and performs loan risk assessments.', 'name': 'risk_agent', 'preferredTransport': 'JSONRPC', 'protocolVersion': '0.3.0', 'skills': [{'description': "Evaluates credit risk scores and performs loan risk assessments. I am a risk analyst. Evaluate the applicant's risk profile u

**risk_agent:**

Based on the financial profile provided, your composite credit risk score is **48 out of 100**, which falls into the **Elevated Risk** category.

### Risk Breakdown:
*   **Credit History:** 20 / 35 (Reflecting a credit score of 680)
*   **Debt-to-Income (DTI):** 8 / 25 (High DTI at 42%)
*   **Loan-to-Value (LTV):** 5 / 20 (High leverage at 0.92)
*   **Derogatory Marks:** 8 / 10 (Impacted by 1 missed payment)
*   **Employment Stability:** 7 / 10 (2 years of current employment)

### Key Risk Flags:
*   **Missed Payments:** You have 1 missed payment recorded in the last 2 years.
*   **High Leverage:** Your LTV of 0.92 and DTI of 42% suggest a higher debt burden relative to your income and asset value.

In [10]:
# -----------------------------------------------------------------------
# Investment query -- should route to the remote investment_agent.
# -----------------------------------------------------------------------

response = await run_agent(
    a2a_coordinator,
    "If I invest 25k now and add 600/mo for 15 years at 7%, what will I have?",
)

05:40:07  INFO      Sending out request, model: gemini-3-flash-preview, backend: GoogleLLMVariant.GEMINI_API, stream: False
05:40:08  INFO      Response received from the model.
05:40:08  INFO      [a2a_coordinator] tool_call: transfer_to_agent({'agent_name': 'investment_agent'})
05:40:08  INFO      [a2a_coordinator] tool_response: transfer_to_agent -> {'result': None}
05:40:09  INFO      HTTP Request: GET http://localhost:8003/.well-known/agent-card.json "HTTP/1.1 200 OK"
05:40:09  INFO      Successfully fetched agent card data from http://localhost:8003/.well-known/agent-card.json: {'capabilities': {}, 'defaultInputModes': ['text/plain'], 'defaultOutputModes': ['text/plain'], 'description': 'Projects investment growth, calculates ROI, and plans savings goals.', 'name': 'investment_agent', 'preferredTransport': 'JSONRPC', 'protocolVersion': '0.3.0', 'skills': [{'description': 'Projects investment growth, calculates ROI, and plans savings goals. I am an investment analyst. Use the avai

**investment_agent:**

If you invest **$25,000** today and contribute **$600 per month** for **15 years** at a **7%** annual return, your investment is projected to grow to approximately **$261,401**.

### Breakdown:
*   **Total Contributions:** $133,000 (Initial $25k + $108k in monthly additions)
*   **Total Interest Earned:** $128,401
*   **Final Balance:** $261,401

The power of compounding is clearly visible here, as your earned interest accounts for nearly half of the total final value!

### 7. Shutdown A2A Servers

In [11]:
# -----------------------------------------------------------------------
# Terminate all background A2A server processes.
# -----------------------------------------------------------------------

for name, proc in processes.items():
    proc.terminate()
    proc.wait(timeout=5)
    log.info("stopped %s agent (pid=%d)", name, proc.pid)

processes.clear()
log.info("all A2A servers stopped")

05:40:18  INFO      stopped mortgage agent (pid=9439)
05:40:19  INFO      stopped risk agent (pid=9440)
05:40:19  INFO      stopped investment agent (pid=9441)
05:40:19  INFO      all A2A servers stopped
